# 01 - Exploración de datos

Recorrido por la tabla canónica `data/interim/matches_unified.csv`: distribución por año, competencia, equipos y fixtures del Mundial 2026.

**Prerequisitos**: haber corrido al menos `import_kaggle_history.py` y `bootstrap_historical_data.py --competitions WC --start-year 2026 --end-year 2026 --no-kaggle`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.data_loader import DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## Carga de la tabla canónica

In [ ]:
matches = DataLoader().load_matches()
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
print(f'Total: {len(matches):,} partidos')
print(f'Rango: {matches["date"].min().date()} -> {matches["date"].max().date()}')
print(f'Columnas: {list(matches.columns)}')

## Distribución por año

In [ ]:
per_year = matches.groupby(matches['date'].dt.year).size()
fig, ax = plt.subplots(figsize=(11, 4))
per_year.plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_title('Partidos por año')
ax.set_xlabel('Año')
ax.set_ylabel('Cantidad de partidos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Distribución por competencia

El cliente Kaggle mapea torneos a códigos canónicos. Histórico tiene `stage="historical"`; Mundial 2026 desde football-data tiene `stage` con `"GROUP_STAGE"` y `group` populado (A..L).

In [ ]:
per_comp = matches['competition'].value_counts()
print(per_comp.to_string())
fig, ax = plt.subplots(figsize=(10, 4))
per_comp.plot(kind='bar', ax=ax, color='#2ca02c')
ax.set_title('Partidos por competencia')
ax.set_ylabel('Cantidad')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Equipos más activos en el histórico

In [ ]:
all_teams = pd.concat([matches['team_a'], matches['team_b']]).dropna()
top20 = all_teams.value_counts().head(20)
print(top20.to_string())

## Fixtures del Mundial 2026 cargadas

Verifica que football-data.org haya populado los grupos correctamente.

In [ ]:
wc2026 = matches[(matches['competition'] == 'WC') & (matches['date'].dt.year == 2026)].copy()
print(f'Fixtures WC 2026: {len(wc2026)}')
print(f'Sin score (por jugar): {wc2026["score_a"].isna().sum()}')
if 'group' in wc2026.columns:
    by_group = wc2026.dropna(subset=['group']).query("group != ''")['group'].value_counts().sort_index()
    print()
    print('Partidos por grupo:')
    print(by_group.to_string())

In [ ]:
if 'group' in wc2026.columns:
    group_teams = (
        wc2026.dropna(subset=['group']).query("group != ''")
        .groupby('group')
        .apply(lambda d: sorted(set(d['team_a'].tolist() + d['team_b'].tolist())))
    )
    for g, teams in group_teams.items():
        print(f'Grupo {g}: {", ".join(teams)}')

## Cobertura por equipos del Mundial 2026 en el histórico

Equipos con muy pocos partidos en el histórico van a tener ratings inestables — buena señal para anticipar predicciones débiles.

In [ ]:
if 'group' in wc2026.columns and len(group_teams) > 0:
    wc_teams = sorted(set(t for teams in group_teams.values for t in teams))
    cov = []
    historical = matches[matches['date'].dt.year < 2026]
    for t in wc_teams:
        n = ((historical['team_a'] == t) | (historical['team_b'] == t)).sum()
        cov.append({'team': t, 'historical_matches': int(n)})
    cov_df = pd.DataFrame(cov).sort_values('historical_matches', ascending=False)
    print(f'Equipos en grupos: {len(wc_teams)}')
    print()
    print('Top 10 con más partidos:')
    print(cov_df.head(10).to_string(index=False))
    print()
    print('Bottom 10 (riesgo de ratings inestables):')
    print(cov_df.tail(10).to_string(index=False))